# Run Trajectory-Aware Sampling Probe

该 notebook 用于 Colab 真实 GPU 环境中的 trajectory-aware sampling scaffold 验证。它只调用仓库 helper / CLI, 不在 notebook 内直接写正式 readiness、selection plan、manifest 或 report。当前验证目标是确认阶段 3 formal 产物可以被读取, 生成下一步真实 GPU runtime 所需的 sampling handoff, 并在提供外部真实 GPU smoke 结果 JSON 时产出 result gate artifact。


## 00 Runtime Identity And User Config


In [ ]:
from pathlib import Path
import os

NOTEBOOK_MANUAL_CONFIG = {
    'repo_url': 'https://github.com/RICHAAARC/TSTW-VW.git',
    'repo_branch': 'main',
    'repo_root': '/content/TSTW-VW',
    'drive_root': '/content/drive/MyDrive/TSTW',
    'stage3_trajectory_root': '',
    'stage3_trajectory_package_path': '',
    'stage3_result_root': '/content/drive/MyDrive/TSTW/results/trajectory_statistic_probe',
    'sampling_config_path': 'configs/protocol/trajectory_aware_sampling_probe.json',
    'result_gate_config_path': 'configs/protocol/trajectory_aware_sampling_real_gpu_backend_connection_smoke_result_gate.json',
    'runtime_validation_gate_config_path': 'configs/protocol/trajectory_aware_sampling_real_backend_runtime_validation_gate.json',
    'explicit_real_generation_transition_decision_config_path': 'configs/protocol/trajectory_aware_sampling_explicit_real_generation_transition_decision.json',
    'external_real_gpu_smoke_results_path': '',
    'backend_connection_probe_config_path': '',
    'run_id_template': 'trajectory_aware_sampling_probe_scaffold_gpu_validation_utc_time_short_commit',
}

REPO_URL = os.environ.get('TSTW_REPO_URL', NOTEBOOK_MANUAL_CONFIG['repo_url'])
REPO_BRANCH = os.environ.get('TSTW_REPO_BRANCH', NOTEBOOK_MANUAL_CONFIG['repo_branch'])
REPO_ROOT = Path(os.environ.get('TSTW_REPO_ROOT', NOTEBOOK_MANUAL_CONFIG['repo_root']))
DRIVE_ROOT = Path(os.environ.get('TSTW_DRIVE_ROOT', NOTEBOOK_MANUAL_CONFIG['drive_root']))
STAGE3_TRAJECTORY_ROOT_TEXT = os.environ.get('TSTW_STAGE3_TRAJECTORY_ROOT', NOTEBOOK_MANUAL_CONFIG['stage3_trajectory_root']).strip()
STAGE3_TRAJECTORY_PACKAGE_PATH_TEXT = os.environ.get('TSTW_STAGE3_TRAJECTORY_PACKAGE_PATH', NOTEBOOK_MANUAL_CONFIG['stage3_trajectory_package_path']).strip()
STAGE3_RESULT_ROOT = Path(os.environ.get('TSTW_STAGE3_RESULT_ROOT', NOTEBOOK_MANUAL_CONFIG['stage3_result_root']))
SAMPLING_CONFIG_PATH = Path(os.environ.get('TSTW_SAMPLING_CONFIG_PATH', NOTEBOOK_MANUAL_CONFIG['sampling_config_path']))
RESULT_GATE_CONFIG_PATH = Path(os.environ.get('TSTW_RESULT_GATE_CONFIG_PATH', NOTEBOOK_MANUAL_CONFIG['result_gate_config_path']))
RUNTIME_VALIDATION_GATE_CONFIG_PATH = Path(os.environ.get('TSTW_RUNTIME_VALIDATION_GATE_CONFIG_PATH', NOTEBOOK_MANUAL_CONFIG['runtime_validation_gate_config_path']))
EXPLICIT_REAL_GENERATION_TRANSITION_DECISION_CONFIG_PATH = Path(os.environ.get('TSTW_EXPLICIT_REAL_GENERATION_TRANSITION_DECISION_CONFIG_PATH', NOTEBOOK_MANUAL_CONFIG['explicit_real_generation_transition_decision_config_path']))
EXTERNAL_REAL_GPU_SMOKE_RESULTS_PATH_TEXT = os.environ.get('TSTW_EXTERNAL_REAL_GPU_SMOKE_RESULTS_PATH', NOTEBOOK_MANUAL_CONFIG['external_real_gpu_smoke_results_path']).strip()
BACKEND_CONNECTION_PROBE_CONFIG_PATH_TEXT = os.environ.get('TSTW_BACKEND_CONNECTION_PROBE_CONFIG_PATH', NOTEBOOK_MANUAL_CONFIG['backend_connection_probe_config_path']).strip()
RUN_ID_TEMPLATE = os.environ.get('TSTW_RUN_ID_TEMPLATE', NOTEBOOK_MANUAL_CONFIG['run_id_template'])
RUN_ID_OVERRIDE = os.environ.get('TSTW_RUN_ID', '').strip()

STAGE3_TRAJECTORY_ROOT = Path(STAGE3_TRAJECTORY_ROOT_TEXT) if STAGE3_TRAJECTORY_ROOT_TEXT else None
STAGE3_TRAJECTORY_PACKAGE_PATH = Path(STAGE3_TRAJECTORY_PACKAGE_PATH_TEXT) if STAGE3_TRAJECTORY_PACKAGE_PATH_TEXT else None
EXTERNAL_REAL_GPU_SMOKE_RESULTS_PATH = Path(EXTERNAL_REAL_GPU_SMOKE_RESULTS_PATH_TEXT) if EXTERNAL_REAL_GPU_SMOKE_RESULTS_PATH_TEXT else None
BACKEND_CONNECTION_PROBE_CONFIG_PATH = Path(BACKEND_CONNECTION_PROBE_CONFIG_PATH_TEXT) if BACKEND_CONNECTION_PROBE_CONFIG_PATH_TEXT else None



## 01 Mount Google Drive


In [ ]:
from google.colab import drive

drive.mount('/content/drive')
assert DRIVE_ROOT.exists(), f'Drive root not found: {DRIVE_ROOT}'



## 02 Clone Or Update Repository


In [ ]:
import subprocess
from datetime import datetime, timezone

if REPO_ROOT.exists():
    subprocess.run(['git', 'fetch', '--all'], cwd=REPO_ROOT, check=True)
    subprocess.run(['git', 'checkout', REPO_BRANCH], cwd=REPO_ROOT, check=True)
    subprocess.run(['git', 'pull', '--ff-only'], cwd=REPO_ROOT, check=True)
else:
    subprocess.run(['git', 'clone', '--branch', REPO_BRANCH, REPO_URL, str(REPO_ROOT)], check=True)

GIT_COMMIT = subprocess.check_output(
    ['git', 'rev-parse', '--short=7', 'HEAD'],
    cwd=REPO_ROOT,
    text=True,
).strip()
UTC_TIME = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
RUN_ID = RUN_ID_OVERRIDE or RUN_ID_TEMPLATE.replace('utc_time', UTC_TIME).replace('short_commit', GIT_COMMIT)
OUTPUT_ROOT = REPO_ROOT / 'outputs' / 'runs' / RUN_ID
LOCAL_INPUT_ROOT = REPO_ROOT / 'outputs' / 'notebook_inputs' / RUN_ID
PACKAGE_ROOT = DRIVE_ROOT / 'results' / 'trajectory_aware_sampling_probe' / RUN_ID / 'packages'

print('GIT_COMMIT =', GIT_COMMIT)
print('UTC_TIME =', UTC_TIME)
print('RUN_ID =', RUN_ID)
print('OUTPUT_ROOT =', OUTPUT_ROOT)
print('LOCAL_INPUT_ROOT =', LOCAL_INPUT_ROOT)
print('PACKAGE_ROOT =', PACKAGE_ROOT)



## 03 Install Runtime Dependencies


In [ ]:
import subprocess

subprocess.run(['python', '-m', 'pip', 'install', '-q', 'pytest'], check=True)



## 04 Verify GPU Runtime


In [ ]:
import subprocess

subprocess.run(['nvidia-smi'], check=True)
print('GPU runtime is visible to this Colab session.')



## 05 Verify Repository Contract


In [ ]:
import os
import subprocess

repo_env = dict(os.environ)
repo_env['PYTHONPATH'] = str(REPO_ROOT) + os.pathsep + repo_env.get('PYTHONPATH', '')
repo_env['PYTHONUTF8'] = '1'
repo_env['PYTHONIOENCODING'] = 'utf-8'
subprocess.run(['python', 'tools/harness/validate_project_contract.py'], cwd=REPO_ROOT, env=repo_env, check=True)



## 06 Run Sampling Scaffold Smoke Tests


In [ ]:
subprocess.run([
    'python', '-m', 'pytest', '-q',
    'tests/functional/test_trajectory_aware_sampling_readiness.py',
    'tests/functional/test_trajectory_aware_sampling_selection_plan.py',
    'tests/functional/test_trajectory_aware_sampling_artifact_builder.py',
    'tests/functional/test_trajectory_aware_sampling_runner.py',
    'tests/functional/test_trajectory_aware_sampling_real_backend_connection_smoke_handoff.py',
    'tests/functional/test_trajectory_aware_sampling_real_gpu_backend_connection_smoke_result_gate.py',
    'tests/constraints/test_trajectory_aware_sampling_contract.py',
], cwd=REPO_ROOT, env=repo_env, check=True)



## 07 Locate Stage Three Trajectory Output


In [ ]:
import sys

sys.path.insert(0, str(REPO_ROOT))
from paper_workflow.notebook_utils import trajectory_aware_sampling_probe_workflow as sampling_workflow

if STAGE3_TRAJECTORY_ROOT is not None:
    upstream_trajectory_root = STAGE3_TRAJECTORY_ROOT
elif STAGE3_TRAJECTORY_PACKAGE_PATH is not None:
    upstream_trajectory_root = sampling_workflow.extract_trajectory_probe_package(
        STAGE3_TRAJECTORY_PACKAGE_PATH,
        LOCAL_INPUT_ROOT,
    )
else:
    try:
        upstream_trajectory_root = sampling_workflow.find_latest_trajectory_probe_root(
            STAGE3_RESULT_ROOT,
        )
    except FileNotFoundError:
        latest_package = sampling_workflow.find_latest_trajectory_probe_package(
            STAGE3_RESULT_ROOT,
        )
        upstream_trajectory_root = sampling_workflow.extract_trajectory_probe_package(
            latest_package,
            LOCAL_INPUT_ROOT,
        )
        print('Extracted stage 3 package:', latest_package)

print('upstream_trajectory_root =', upstream_trajectory_root)
assert (upstream_trajectory_root / 'records' / 'event_scores.jsonl').exists()
assert (upstream_trajectory_root / 'artifacts' / 'trajectory_mechanism_decision.json').exists()



## 08 Run Sampling Scaffold Validation


In [ ]:
policy_manifest = sampling_workflow.run_sampling_scaffold_cli(
    repository_root=REPO_ROOT,
    upstream_trajectory_root=upstream_trajectory_root,
    output_root=OUTPUT_ROOT,
    sampling_config_path=SAMPLING_CONFIG_PATH,
)
gpu_validation_contract = sampling_workflow.read_gpu_validation_contract(OUTPUT_ROOT)
backend_transition_guard = sampling_workflow.read_backend_transition_guard(OUTPUT_ROOT)
backend_transition_decision = sampling_workflow.read_backend_transition_decision(OUTPUT_ROOT)
runtime_interface_scaffold = sampling_workflow.read_runtime_interface_scaffold(OUTPUT_ROOT)
runtime_interface_implementation = sampling_workflow.read_runtime_interface_implementation(OUTPUT_ROOT)
backend_integration_decision = sampling_workflow.read_backend_integration_decision(OUTPUT_ROOT)
backend_adapter_scaffold = sampling_workflow.read_backend_adapter_scaffold(OUTPUT_ROOT)
backend_connection_contract = sampling_workflow.read_backend_connection_contract(OUTPUT_ROOT)
real_backend_connection_smoke = sampling_workflow.read_real_backend_connection_smoke(OUTPUT_ROOT)
real_backend_connection_smoke_handoff = sampling_workflow.read_real_backend_connection_smoke_handoff(OUTPUT_ROOT)
print(policy_manifest)
print(gpu_validation_contract)
print(backend_transition_guard)
print(backend_transition_decision)
print(runtime_interface_scaffold)
print(runtime_interface_implementation)
print(backend_integration_decision)
print(backend_adapter_scaffold)
print(backend_connection_contract)
print(real_backend_connection_smoke)
print(real_backend_connection_smoke_handoff)


## 09 Run Real GPU Backend Connection Smoke Result Gate


In [ ]:
external_real_gpu_smoke_results_path = EXTERNAL_REAL_GPU_SMOKE_RESULTS_PATH
backend_connection_probe_config_path = BACKEND_CONNECTION_PROBE_CONFIG_PATH
if backend_connection_probe_config_path is None:
    backend_connection_probe_config_path = sampling_workflow.write_default_backend_connection_probe_config(
        OUTPUT_ROOT / 'artifacts' / 'backend_connection_probe_config.json',
    )
    print('Generated default non-generative backend connection probe config:')
    print(backend_connection_probe_config_path)

if external_real_gpu_smoke_results_path is None:
    external_real_gpu_smoke_results_path = OUTPUT_ROOT / 'artifacts' / 'external_real_gpu_smoke_results.json'
    external_real_gpu_smoke_results = sampling_workflow.write_environment_only_real_gpu_backend_connection_smoke_results(
        repository_root=REPO_ROOT,
        output_path=external_real_gpu_smoke_results_path,
        backend_connection_probe_config_path=backend_connection_probe_config_path,
    )
    print('Generated environment-only external real GPU smoke results:')
    print(external_real_gpu_smoke_results)
else:
    external_real_gpu_smoke_results = sampling_workflow.read_external_real_gpu_backend_connection_smoke_results(
        external_real_gpu_smoke_results_path,
    )
    print('Loaded external real GPU smoke results:')
    print(external_real_gpu_smoke_results)

real_gpu_backend_connection_smoke_result_gate = sampling_workflow.run_real_gpu_backend_connection_smoke_result_gate(
    repository_root=REPO_ROOT,
    run_root=OUTPUT_ROOT,
    external_smoke_results_path=external_real_gpu_smoke_results_path,
    result_gate_config_path=RESULT_GATE_CONFIG_PATH,
)
print(real_gpu_backend_connection_smoke_result_gate)

real_backend_runtime_validation_gate = sampling_workflow.run_real_backend_runtime_validation_gate(
    repository_root=REPO_ROOT,
    run_root=OUTPUT_ROOT,
    runtime_validation_gate_config_path=RUNTIME_VALIDATION_GATE_CONFIG_PATH,
)
print(real_backend_runtime_validation_gate)

explicit_real_generation_transition_decision = sampling_workflow.run_explicit_real_generation_transition_decision(
    repository_root=REPO_ROOT,
    run_root=OUTPUT_ROOT,
    transition_decision_config_path=EXPLICIT_REAL_GENERATION_TRANSITION_DECISION_CONFIG_PATH,
)
print(explicit_real_generation_transition_decision)


## 10 Package Sampling Results


In [ ]:
archive_path = sampling_workflow.package_sampling_probe_run(
    run_root=OUTPUT_ROOT,
    package_root=PACKAGE_ROOT,
    package_name=RUN_ID,
)
print('archive_path =', archive_path)



## 11 Print Final Summary


In [ ]:
from pathlib import Path

print('SamplingReadinessDecision =', policy_manifest.get('SamplingReadinessDecision'))
print('SamplingSelectionPlanDecision =', policy_manifest.get('SamplingSelectionPlanDecision'))
print('selected_sampling_policy_kind =', policy_manifest.get('selected_sampling_policy_kind'))
print('selected_record_count =', policy_manifest.get('selected_record_count'))
print('requires_real_gpu_validation =', policy_manifest.get('requires_real_gpu_validation'))
print('next_step_requires_real_gpu_validation =', policy_manifest.get('next_step_requires_real_gpu_validation'))
print('NextRequiredValidationBySampling =', policy_manifest.get('NextRequiredValidationBySampling'))
print('TrajectoryAwareSamplingGpuValidationContractDecision =', gpu_validation_contract.get('TrajectoryAwareSamplingGpuValidationContractDecision'))
print('NextAllowedConstructionAfterGpuValidationContract =', gpu_validation_contract.get('NextAllowedConstructionAfterGpuValidationContract'))
print('TrajectoryAwareSamplingBackendTransitionGuardDecision =', backend_transition_guard.get('TrajectoryAwareSamplingBackendTransitionGuardDecision'))
print('NextAllowedConstructionAfterBackendTransitionGuard =', backend_transition_guard.get('NextAllowedConstructionAfterBackendTransitionGuard'))
print('TrajectoryAwareSamplingBackendTransitionDecision =', backend_transition_decision.get('TrajectoryAwareSamplingBackendTransitionDecision'))
print('NextAllowedConstructionAfterBackendTransitionDecision =', backend_transition_decision.get('NextAllowedConstructionAfterBackendTransitionDecision'))
print('TrajectoryAwareSamplingRuntimeInterfaceScaffoldDecision =', runtime_interface_scaffold.get('TrajectoryAwareSamplingRuntimeInterfaceScaffoldDecision'))
print('NextAllowedConstructionAfterRuntimeInterfaceScaffold =', runtime_interface_scaffold.get('NextAllowedConstructionAfterRuntimeInterfaceScaffold'))
print('request_prototype_count =', runtime_interface_scaffold.get('request_prototype_count'))
print('runtime_interface_scaffold_allowed =', runtime_interface_scaffold.get('runtime_interface_scaffold_allowed'))
print('TrajectoryAwareSamplingRuntimeInterfaceImplementationDecision =', runtime_interface_implementation.get('TrajectoryAwareSamplingRuntimeInterfaceImplementationDecision'))
print('NextAllowedConstructionAfterRuntimeInterfaceImplementation =', runtime_interface_implementation.get('NextAllowedConstructionAfterRuntimeInterfaceImplementation'))
print('dry_run_request_count =', runtime_interface_implementation.get('dry_run_request_count'))
print('dry_run_result_manifest_count =', runtime_interface_implementation.get('dry_run_result_manifest_count'))
print('runtime_interface_implementation_allowed =', runtime_interface_implementation.get('runtime_interface_implementation_allowed'))
print('TrajectoryAwareSamplingBackendIntegrationDecision =', backend_integration_decision.get('TrajectoryAwareSamplingBackendIntegrationDecision'))
print('NextAllowedConstructionAfterBackendIntegrationDecision =', backend_integration_decision.get('NextAllowedConstructionAfterBackendIntegrationDecision'))
print('backend_adapter_scaffold_allowed =', backend_integration_decision.get('backend_adapter_scaffold_allowed'))
print('TrajectoryAwareSamplingBackendAdapterScaffoldDecision =', backend_adapter_scaffold.get('TrajectoryAwareSamplingBackendAdapterScaffoldDecision'))
print('NextAllowedConstructionAfterBackendAdapterScaffold =', backend_adapter_scaffold.get('NextAllowedConstructionAfterBackendAdapterScaffold'))
print('adapter_schema_count =', backend_adapter_scaffold.get('adapter_schema_count'))
print('adapter_stub_count =', backend_adapter_scaffold.get('adapter_stub_count'))
print('TrajectoryAwareSamplingBackendConnectionContractDecision =', backend_connection_contract.get('TrajectoryAwareSamplingBackendConnectionContractDecision'))
print('NextAllowedConstructionAfterBackendConnectionContract =', backend_connection_contract.get('NextAllowedConstructionAfterBackendConnectionContract'))
print('real_backend_connection_smoke_allowed_after_contract =', backend_connection_contract.get('real_backend_connection_smoke_allowed_after_contract'))
print('contract_section_count =', backend_connection_contract.get('contract_section_count'))
print('TrajectoryAwareSamplingRealBackendConnectionSmokeDecision =', real_backend_connection_smoke.get('TrajectoryAwareSamplingRealBackendConnectionSmokeDecision'))
print('NextRequiredValidationAfterRealBackendConnectionSmokeRequest =', real_backend_connection_smoke.get('NextRequiredValidationAfterRealBackendConnectionSmokeRequest'))
print('gpu_execution_required_for_next_step =', real_backend_connection_smoke.get('gpu_execution_required_for_next_step'))
print('real_backend_connection_attempted =', real_backend_connection_smoke.get('real_backend_connection_attempted'))
print('TrajectoryAwareSamplingRealBackendConnectionSmokeHandoffDecision =', real_backend_connection_smoke_handoff.get('TrajectoryAwareSamplingRealBackendConnectionSmokeHandoffDecision'))
print('NextRequiredExternalExecutionAfterSmokeHandoff =', real_backend_connection_smoke_handoff.get('NextRequiredExternalExecutionAfterSmokeHandoff'))
print('external_gpu_required =', real_backend_connection_smoke_handoff.get('external_gpu_required'))
print('real_backend_connection_smoke_handoff_allowed =', real_backend_connection_smoke_handoff.get('real_backend_connection_smoke_handoff_allowed'))
print('TrajectoryAwareSamplingRealGpuBackendConnectionSmokeResultGateDecision =', real_gpu_backend_connection_smoke_result_gate.get('TrajectoryAwareSamplingRealGpuBackendConnectionSmokeResultGateDecision'))
print('NextAllowedConstructionAfterRealGpuBackendConnectionSmokeResultGate =', real_gpu_backend_connection_smoke_result_gate.get('NextAllowedConstructionAfterRealGpuBackendConnectionSmokeResultGate'))
print('TrajectoryAwareSamplingRealBackendRuntimeValidationGateDecision =', real_backend_runtime_validation_gate.get('TrajectoryAwareSamplingRealBackendRuntimeValidationGateDecision'))
print('NextAllowedConstructionAfterRealBackendRuntimeValidationGate =', real_backend_runtime_validation_gate.get('NextAllowedConstructionAfterRealBackendRuntimeValidationGate'))
print('adapter_schema_validation_status =', real_backend_runtime_validation_gate.get('adapter_schema_validation_status'))
print('failure_path_validation_status =', real_backend_runtime_validation_gate.get('failure_path_validation_status'))
print('TrajectoryAwareSamplingExplicitRealGenerationTransitionDecision =', explicit_real_generation_transition_decision.get('TrajectoryAwareSamplingExplicitRealGenerationTransitionDecision'))
print('NextAllowedConstructionAfterExplicitRealGenerationTransitionDecision =', explicit_real_generation_transition_decision.get('NextAllowedConstructionAfterExplicitRealGenerationTransitionDecision'))
print('controlled_request_scaffold_allowed =', explicit_real_generation_transition_decision.get('controlled_request_scaffold_allowed'))
print('maximum_controlled_request_count =', explicit_real_generation_transition_decision.get('maximum_controlled_request_count'))
print('observed_result_artifact_count =', real_gpu_backend_connection_smoke_result_gate.get('observed_result_artifact_count'))
print('missing_required_result_artifact_kinds =', real_gpu_backend_connection_smoke_result_gate.get('missing_required_result_artifact_kinds'))
print('External real GPU smoke results path:', external_real_gpu_smoke_results_path)
print('Backend connection probe config path:', backend_connection_probe_config_path)
print('backend_transition_decision_required =', backend_transition_guard.get('backend_transition_decision_required'))
print('real_generation_allowed =', policy_manifest.get('real_generation_allowed'))
print('real_watermark_integration_allowed =', policy_manifest.get('real_watermark_integration_allowed'))
print('Download package:', archive_path)
print('Report path:', OUTPUT_ROOT / 'reports' / 'trajectory_aware_sampling_probe_report.md')
print('GPU validation contract path:', OUTPUT_ROOT / 'artifacts' / 'trajectory_aware_sampling_gpu_validation_contract.json')
print('Backend transition guard path:', OUTPUT_ROOT / 'artifacts' / 'trajectory_aware_sampling_backend_transition_guard.json')
print('Backend transition decision path:', OUTPUT_ROOT / 'artifacts' / 'trajectory_aware_sampling_backend_transition_decision.json')
print('Runtime interface scaffold path:', OUTPUT_ROOT / 'artifacts' / 'trajectory_aware_sampling_runtime_interface_scaffold.json')
print('Runtime interface implementation path:', OUTPUT_ROOT / 'artifacts' / 'trajectory_aware_sampling_runtime_interface_implementation.json')
print('Backend integration decision path:', OUTPUT_ROOT / 'artifacts' / 'trajectory_aware_sampling_backend_integration_decision.json')
print('Backend adapter scaffold path:', OUTPUT_ROOT / 'artifacts' / 'trajectory_aware_sampling_backend_adapter_scaffold.json')
print('Backend connection contract path:', OUTPUT_ROOT / 'artifacts' / 'trajectory_aware_sampling_backend_connection_contract.json')
print('Real backend connection smoke path:', OUTPUT_ROOT / 'artifacts' / 'trajectory_aware_sampling_real_backend_connection_smoke.json')
print('Real backend connection smoke handoff path:', OUTPUT_ROOT / 'artifacts' / 'trajectory_aware_sampling_real_backend_connection_smoke_handoff.json')
print('Real GPU backend connection smoke result gate path:', OUTPUT_ROOT / 'artifacts' / 'trajectory_aware_sampling_real_gpu_backend_connection_smoke_result_gate.json')
print('Real backend runtime validation gate path:', OUTPUT_ROOT / 'artifacts' / 'trajectory_aware_sampling_real_backend_runtime_validation_gate.json')
print('Explicit real generation transition decision path:', OUTPUT_ROOT / 'artifacts' / 'trajectory_aware_sampling_explicit_real_generation_transition_decision.json')
